# ADMET multi-endpoint prediction (DeepFP PKL → wide CSV)

Load a **merged** fingerprint PKL (same contract as `src/predict_arcmol_from_fp_pkls.py`), then run label-free inference for **every** `checkpoints/<task_folder>/` that contains a matching pair of `*.bundle.pt` and same-stem `*.pth`. The result is one CSV row per compound and one or more columns per endpoint.

- **~73 ADMET endpoints**: see `configs/tasks_template_admet.csv` (~73 rows). Each `task_name` should have an exported **bundle + weights** under `checkpoints/<task_name>/` (folder name matches the template).
- This repo may ship only a subset of checkpoints; the notebook still **iterates every valid job** found on disk. Missing folders or incomplete pairs are simply skipped.
- **`EXCLUDE_PREFIXES = ()`** below means **no** folder is filtered (including `benchmark_*` demos). Set e.g. `("benchmark_",)` if you want ADMET-only columns.

**Populate `checkpoints/`**: if you keep the trained bundle at `CMD-ADMET/ARCMOL_ADMET_BEST/checkpoints/` next to this repo tree, run once from repo root:  
`bash scripts/sync_admet_checkpoints_from_best.sh`  
(Or pass the source directory as the first argument.) This **merges** into `checkpoints/`; existing folders (e.g. `benchmark_*`) are kept.

**Pack checkpoints for release**:  
`bash scripts/pack_admet_checkpoints.sh [optional_output.tar.gz]`  
Archives `checkpoints/` while excluding `benchmark_*` by default.

**For manuscript reviewers:** after **Kernel → Restart & Run All**, the inference cell **renders tables and the full list of endpoint columns inside this notebook** (not only the CSV). Commit or export the executed `.ipynb` / HTML so reviewers see the embedded outputs.


## Environment

Requires PyTorch, scikit-learn, and pandas. **GPU is recommended.** The working directory is set to the repository root.

**Stale outputs:** If you open this file and still see only ~2 `[OK] ...` lines under the prediction cell, that is **old saved output** from a previous run (or a clone where `checkpoints/` had only two models). Use **Restart kernel & run all** after syncing full weights. The number of tasks printed as `[all_checkpoints] jobs=N` must match how many valid `*.bundle.pt` + `.pth` pairs exist under `checkpoints/`.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
import torch

REPO_ROOT = Path("..").resolve()
os.chdir(REPO_ROOT)
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from predict_arcmol_from_fp_pkls import (
    discover_checkpoint_jobs,
    filter_checkpoint_jobs,
    load_merged_fp_pkl_file,
    run_all_checkpoints_wide,
)

# Merged DeepFP PKL (change to your path)
PKL_FILE = REPO_ROOT / "data" / "_demo_pkls_output" / "all_batch_0_3.pkl"
CHECKPOINTS_ROOT = REPO_ROOT / "checkpoints"
OUT_CSV = REPO_ROOT / "predictions" / "admet_all_endpoints_preds.csv"
TASK_LIST_CSV = REPO_ROOT / "configs" / "tasks_template_admet.csv"

# Empty tuple = predict for every subfolder that has bundle+.pth pairs (including benchmark_*).
# Use ("benchmark_",) to skip MoleculeNet demo checkpoints.
EXCLUDE_PREFIXES: tuple[str, ...] = ()
BATCH_SIZE = 64

assert PKL_FILE.is_file(), PKL_FILE
assert CHECKPOINTS_ROOT.is_dir(), CHECKPOINTS_ROOT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("REPO_ROOT:", REPO_ROOT)
print("Device:", device)
print("PKL_FILE:", PKL_FILE)
print("EXCLUDE_PREFIXES:", EXCLUDE_PREFIXES or "(none — all folders scanned)")


/home/zyrlia/zou/Manba/Mamba/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPO_ROOT: /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main
Device: cpu
PKL_FILE: /home/data/zou/CMD-/CMD-ADMET/ARCMOL_github/ArcMol-main/data/_demo_pkls_output/all_batch_0_3.pkl
EXCLUDE_PREFIXES: (none — all folders scanned)


In [2]:
merged = load_merged_fp_pkl_file(str(PKL_FILE))
print("Samples in PKL:", len(merged))
_keys = list(merged[0].keys())
print("PKL keys (count):", len(_keys), "| first few:", _keys[:8])


Samples in PKL: 3
PKL keys (count): 17 | first few: ['SMILES', 'AttrMask', 'BioT5', 'EStateFingerprint', 'GPT-GNN', 'GROVER', 'GraphCL', 'GraphMVP']


In [3]:
tasks_df = pd.read_csv(TASK_LIST_CSV)
print("ADMET template rows:", len(tasks_df))
print("Example task_name:", tasks_df["task_name"].head(5).tolist(), "...")


ADMET template rows: 73
Example task_name: ['ames', 'avian_tox', 'bbb_cns', 'bbb_logbb', 'bcrp'] ...


In [4]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

_n_jobs = len(
    filter_checkpoint_jobs(
        discover_checkpoint_jobs(str(CHECKPOINTS_ROOT)),
        EXCLUDE_PREFIXES,
    )
)
print(f"Checkpoint jobs to run: {_n_jobs} (if this is 2, sync checkpoints/ or fix EXCLUDE_PREFIXES)")

df_out = run_all_checkpoints_wide(
    merged,
    str(CHECKPOINTS_ROOT),
    str(OUT_CSV),
    extra_attrs=["SMILES"],
    batch_size=BATCH_SIZE,
    device=device,
    exclude_dir_prefixes=EXCLUDE_PREFIXES,
)
print("Output:", OUT_CSV)
print("Result shape:", df_out.shape)
_cc = list(df_out.columns)
print("Columns: first", _cc[:5], "... last", _cc[-3:])

# --- Embedded tables for reviewers (saved inside this .ipynb when you Run All) ---
from IPython.display import Markdown, display

display(Markdown("---"))
display(Markdown("### Results embedded for reviewers / supplementary material"))
display(
    Markdown(
        f"- **Compounds (rows):** {len(df_out)}  \n"
        f"- **Columns:** {len(df_out.columns)}  \n"
        f"- **CSV (full wide table):** `{OUT_CSV}`  \n"
        f"- Wide tables are split into **left** and **right** blocks; all prediction column names are listed at the bottom."
    )
)

_edge = 12
if len(_cc) <= _edge * 2:
    with pd.option_context("display.max_columns", None, "display.width", 200, "display.max_colwidth", 30):
        display(df_out)
else:
    display(Markdown("**Preview — left columns (SMILES + first endpoints):**"))
    with pd.option_context("display.max_columns", None, "display.width", 200, "display.max_colwidth", 30):
        display(df_out.loc[:, _cc[:_edge]])
    display(Markdown("**Preview — right columns (last endpoints):**"))
    with pd.option_context("display.max_columns", None, "display.width", 200, "display.max_colwidth", 30):
        display(df_out.loc[:, _cc[-_edge:]])

_pred_cols = [c for c in _cc if c != "SMILES"]
display(Markdown(f"**All {len(_pred_cols)} prediction-related column names:**"))
display(pd.DataFrame({"column": _pred_cols}))


Checkpoint jobs to run: 77 (if this is 2, sync checkpoints/ or fix EXCLUDE_PREFIXES)
[all_checkpoints] jobs=77  exclude_prefixes=none


/home/zyrlia/zou/Manba/Mamba/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[OK] CHEMBL2147_Ki (CHEMBL2147_Ki)  reg


/home/zyrlia/zou/Manba/Mamba/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[OK] ames (ames)  cls


...

,SMILES,CHEMBL2147_Ki_pred,ames_pred_prob,ames_pred_label,avian_tox_pred_prob,avian_tox_pred_label,bbb_cns_pred,bbb_logbb_pred_prob,bbb_logbb_pred_label,bcrp_pred_prob,bcrp_pred_label,bee_tox_pred_prob
0,CCO,-2.052860,0.014427,0.0,0.054943,0.0,-2.516206,0.727575,1.0,0.023030,0.0,0.495810
1,Cc1ccccc1O,-1.419269,0.141964,0.0,0.030562,0.0,-1.715790,0.739985,1.0,0.138743,0.0,0.501428
2,CC(C)CC(C)C,-1.273080,0.041023,0.0,0.021903,0.0,-2.164368,0.949304,1.0,0.096825,0.0,0.498560


**Preview — right columns (last endpoints):**

,sr_are_pred_label,sr_atad5_pred_prob,sr_atad5_pred_label,sr_hse_pred_prob,sr_hse_pred_label,sr_mmp_pred_prob,sr_mmp_pred_label,sr_p53_pred_prob,sr_p53_pred_label,t0.5_pred_prob,t0.5_pred_label,vd_pred
0,0.0,0.004037,0.0,0.053254,0.0,0.026416,0.0,0.009618,0.0,0.309071,0.0,-0.053252
1,0.0,0.003107,0.0,0.041455,0.0,0.045943,0.0,0.051700,0.0,0.452668,0.0,0.140847
2,0.0,0.001169,0.0,0.026627,0.0,0.019501,0.0,0.029411,0.0,0.376692,0.0,0.595077


**All 128 prediction-related column names:**

,column
0,CHEMBL2147_Ki_pred
1,ames_pred_prob
2,ames_pred_label
3,avian_tox_pred_prob
4,avian_tox_pred_label
...,...
123,sr_p53_pred_prob
124,sr_p53_pred_label
125,t0.5_pred_prob
126,t0.5_pred_label


### CLI equivalent

```bash
cd /path/to/ArcMol-main
python src/predict_arcmol_from_fp_pkls.py \
  --pkl_file data/_demo_pkls_output/all_batch_0_3.pkl \
  --checkpoints_root checkpoints \
  --out_dir predictions \
  --out_csv admet_all_endpoints_preds.csv \
  --no_skip_benchmark
```

`--no_skip_benchmark` matches `EXCLUDE_PREFIXES = ()` in this notebook (predict every valid checkpoint folder). Omit it to skip `benchmark_*` by default.


### What to upload to GitHub

**Commit in the repo (code + docs + small configs)**  
- `notebook/admet_predict_from_fp_pkl.ipynb` — **for peer review**, prefer the **executed** version (with outputs saved) or export **File → Save and Export Notebook As → HTML** so reviewers see the embedded tables without re-running.  
- `src/predict_arcmol_from_fp_pkls.py` (and any other `src/` modules it imports that you changed)  
- `scripts/sync_admet_checkpoints_from_best.sh` (optional local setup)  
- `scripts/pack_admet_checkpoints.sh`  
- `configs/tasks_template_admet.csv`  
- `.gitignore` (includes `predictions/` so local CSVs are not committed)

**Do not commit (large or regenerated)**  
- `predictions/*.csv` — outputs; regenerate with the notebook or CLI.  
- Full `checkpoints/` tree — often too large; use the pack script + **GitHub Release** (attach `admet_checkpoints_bundle.tar.gz`) or Git LFS.

**Release asset (optional but practical)**  
- Run `./scripts/pack_admet_checkpoints.sh` and upload the resulting `.tar.gz` as a release attachment so users can extract `checkpoints/` next to the clone.

**Demo data (optional)**  
- Small demo PKLs under `data/_demo_pkls_output/` if you want the notebook to run out of the box; otherwise document that users must point `PKL_FILE` to their merged DeepFP output.